<center><h1>Kim_Beomjun_HW3</h1></center>
<br>
<br>

Name: Beomjun Kim 
<br>
Github Username: valen991693
<br>
USC ID: 4391-4935-38

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO

Get the AReM Data Set

In [ ]:
DATA_DIR = Path("../data")

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

for activity in activities:
    files = sorted((DATA_DIR / activity).glob("dataset*.csv"))
    print(f"{activity}: {len(files)} datasets")

bending1: 7 datasets
bending2: 6 datasets
cycling: 15 datasets
lying: 15 datasets
sitting: 15 datasets
standing: 15 datasets
walking: 15 datasets


In [3]:
total_datasets = sum(
    len(list((DATA_DIR / activity).glob("dataset*.csv")))
    for activity in activities
)

print("Total number of datasets:", total_datasets)

Total number of datasets: 88


### (b) Test and Train Data

In [4]:
train_files = []
test_files = []

for activity in activities:
    activity_dir = DATA_DIR / activity
    files = sorted(
        activity_dir.glob("dataset*.csv"),
        key=lambda x: int(x.stem.replace("dataset", ""))
    )

    for file_path in files:
        dataset_num = int(file_path.stem.replace("dataset", ""))

        if activity in ["bending1", "bending2"]:
            if dataset_num in [1, 2]:
                test_files.append((activity, file_path))
            else:
                train_files.append((activity, file_path))

        else:
            if dataset_num in [1, 2, 3]:
                test_files.append((activity, file_path))
            else:
                train_files.append((activity, file_path))

In [5]:
print("Number of training instances:", len(train_files))
print("Number of test instances:", len(test_files))
print("Total:", len(train_files) + len(test_files))

Number of training instances: 69
Number of test instances: 19
Total: 88


In [6]:
train_counts = {}
test_counts = {}

for activity in activities:
    train_counts[activity] = sum(
        1 for a, _ in train_files if a == activity
    )
    test_counts[activity] = sum(
        1 for a, _ in test_files if a == activity
    )

split_summary = pd.DataFrame({
    "Train": train_counts,
    "Test": test_counts
})

split_summary["Total"] = (
    split_summary["Train"] + split_summary["Test"]
)

split_summary

,Train,Test,Total
bending1,5,2,7
bending2,4,2,6
cycling,12,3,15
lying,12,3,15
sitting,12,3,15
standing,12,3,15
walking,12,3,15


### (c) Feature Extraction

#### i. Research

Common time-domain features used in time series classification include:

- Minimum and maximum
- Mean
- Median
- First and third quartiles
- Standard deviation and variance
- Range
- Interquartile range (IQR)
- Root mean square (RMS)
- Skewness
- Kurtosis
- Mean absolute deviation
- Zero-crossing rate

#### ii. Extraction

##### Data Cleaning

First, all data files were checked to determine whether they could be parsed using the expected seven-column structure.

In [7]:
columns = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

In [8]:
parsing_errors = []

for split, file_list in [("train", train_files), ("test", test_files)]:
    for activity, file_path in file_list:
        try:
            pd.read_csv(
                file_path,
                comment="#",
                header=None,
                names=columns
            )
        except pd.errors.ParserError as e:
            parsing_errors.append(
                (activity, file_path.name, str(e))
            )

parsing_errors

[('cycling',
  'dataset9.csv',
  'Error tokenizing data. C error: Expected 7 fields in line 485, saw 8\n'),
 ('cycling',
  'dataset14.csv',
  'Error tokenizing data. C error: Expected 7 fields in line 485, saw 8\n')]

The parsing errors occurred in `cycling/dataset9.csv` and `cycling/dataset14.csv`. 

The final observations of these files were inspected to identify the cause.

In [9]:
for dataset in ["dataset9.csv", "dataset14.csv"]:
    file_path = DATA_DIR / "cycling" / dataset

    with open(file_path, "r") as f:
        last_line = f.readlines()[-1]

    print(f"{dataset}: {repr(last_line)}")

dataset9.csv: '119750,38.33,0.94,15.25,2.17,20.33,1.25,'
dataset14.csv: '119750,34.00,2.12,24.50,1.80,22.25,2.59,'


Both files contain an extra trailing comma in their final observations.
This creates an additional empty field, causing eight fields to be detected instead of the expected seven.

An additional formatting inconsistency was found in `bending2/dataset4.csv`. Unlike the other data files, its observations are separated by whitespace rather than commas.

In [10]:
problem_path = DATA_DIR / "bending2" / "dataset4.csv"

with open(problem_path, "r") as f:
    lines = f.readlines()

for line in lines[:7]:
    print(repr(line))

'# Task: bending2\n'
'# Frequency (Hz): 20\n'
'# Clock (millisecond): 250\n'
'# Duration (seconds): 120\n'
'# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23\n'
'0 32.50 0.50 0.00 0.00 19.00 1.00 \n'
'250 32.50 0.50 0.00 0.00 18.50 0.50 \n'


To handle these formatting inconsistencies consistently without modifying the original files, a data-reading function was created. It removes metadata lines and trailing commas and converts whitespace-separated observations to the same comma-separated format used by the other files.

In [11]:
def read_arem_file(file_path):
    cleaned_lines = []

    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue

            line = line.rstrip().rstrip(",")

            # Convert whitespace-separated rows to comma-separated rows
            if "," not in line:
                line = ",".join(line.split())

            cleaned_lines.append(line)

    return pd.read_csv(
        StringIO("\n".join(cleaned_lines)),
        header=None,
        names=columns
    )

After resolving the formatting inconsistencies, all data files were checked against the expected dimensions of 480 observations and seven columns.

In [12]:
irregular_files = []

for split, file_list in [("train", train_files), ("test", test_files)]:
    for activity, file_path in file_list:
        df = read_arem_file(file_path)

        if df.shape != (480, 7):
            irregular_files.append(
                (activity, file_path.name, df.shape)
            )

irregular_files

[('sitting', 'dataset8.csv', (479, 7))]

`sitting/dataset8.csv` contains 479 observations rather than the expected 480. Its timestamps were inspected to identify the missing observation.

In [13]:
problem_path = DATA_DIR / "sitting" / "dataset8.csv"
df_problem = read_arem_file(problem_path)

df_problem.loc[52:54, ["time"]]

,time
52,13000
53,13250
54,13750


The timestamps normally increase by 250 ms. In `sitting/dataset8.csv`, the sequence jumps from 13250 ms to 13750 ms, indicating that the observation at 13500 ms is missing.

Since the original sensor measurements at this timestamp are unavailable, no artificial observation was imputed. The 479 available observations were therefore used for feature extraction.

##### Feature Extraction

For each instance, the minimum, maximum, mean, median, standard deviation, first quartile, and third quartile were extracted from each of the six time series. This produces 42 time-domain features per instance.

In [14]:
def extract_features(df):
    features = {}

    time_series = [
        "avg_rss12",
        "var_rss12",
        "avg_rss13",
        "var_rss13",
        "avg_rss23",
        "var_rss23"
    ]

    for i, column in enumerate(time_series, start=1):
        series = df[column]

        features[f"min{i}"] = series.min()
        features[f"max{i}"] = series.max()
        features[f"mean{i}"] = series.mean()
        features[f"median{i}"] = series.median()
        features[f"std{i}"] = series.std()
        features[f"1st_quart{i}"] = series.quantile(0.25)
        features[f"3rd_quart{i}"] = series.quantile(0.75)

    return features

In [15]:
feature_rows = []

for split, file_list in [("train", train_files), ("test", test_files)]:
    for activity, file_path in file_list:
        df = read_arem_file(file_path)

        features = extract_features(df)

        features["activity"] = activity
        features["dataset"] = file_path.stem
        features["split"] = split

        feature_rows.append(features)

feature_df = pd.DataFrame(feature_rows)

feature_df

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,dataset,split
0,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,0.0,1.79,0.493292,0.430,0.513506,0.000,0.940,bending1,dataset3,train
1,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,0.0,2.18,0.613521,0.500,0.524317,0.000,1.000,bending1,dataset4,train
2,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,0.0,1.79,0.383292,0.430,0.389164,0.000,0.500,bending1,dataset5,train
3,37.00,48.00,43.454958,43.250,1.386098,42.50,45.0000,0.0,1.58,0.378083,...,0.0,5.26,0.679646,0.500,0.622534,0.430,0.870,bending1,dataset6,train
4,36.25,48.00,43.969125,44.500,1.618364,43.31,44.6700,0.0,1.50,0.413125,...,0.0,2.96,0.555312,0.490,0.487826,0.000,0.830,bending1,dataset7,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,35.50,46.25,43.174938,43.670,1.989052,42.50,44.5000,0.0,2.12,0.506583,...,0.0,5.72,0.911979,0.830,0.666161,0.470,1.220,standing,dataset2,test
84,32.75,47.00,42.760562,44.500,3.398919,41.33,45.3725,0.0,3.34,0.486167,...,0.0,5.73,0.842271,0.710,0.722165,0.430,1.090,standing,dataset3,test
85,19.33,43.50,34.227771,35.500,4.889576,30.50,37.7500,0.0,14.50,3.995729,...,0.0,9.74,3.394125,3.100,1.792090,2.105,4.425,walking,dataset1,test
86,12.50,45.00,33.509729,34.125,4.850923,30.50,36.7500,0.0,13.05,4.450771,...,0.0,8.96,3.378479,3.085,1.787360,2.060,4.440,walking,dataset2,test


The extracted feature data were checked for the expected number of instances, missing values, and the previously defined training/test split.

In [16]:
print("Feature DataFrame shape:", feature_df.shape)
print("Missing values:", feature_df.isna().sum().sum())
print()
print(feature_df["split"].value_counts())

Feature DataFrame shape: (88, 45)
Missing values: 0

split
train    69
test     19
Name: count, dtype: int64


#### iii. Standard Deviation

In [17]:
feature_columns = [
    col for col in feature_df.columns
    if col not in ["activity", "dataset", "split"]
]

print("Number of features:", len(feature_columns))

Number of features: 42


For each of the 42 extracted time-domain features, the sample standard deviation was estimated across the 88 instances. A 90% confidence interval for each standard deviation was constructed using percentile bootstrap resampling with 10,000 bootstrap samples. A fixed random seed was used for reproducibility.

In [18]:
def bootstrap_std_ci(values, n_bootstrap=10000, confidence=0.90, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    bootstrap_std = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True
        )

        bootstrap_std[i] = np.std(sample, ddof=1)

    alpha = 1 - confidence

    lower = np.percentile(
        bootstrap_std,
        100 * alpha / 2
    )

    upper = np.percentile(
        bootstrap_std,
        100 * (1 - alpha / 2)
    )

    return lower, upper

In [19]:
rng = np.random.default_rng(42)

results = []

for feature in feature_columns:
    values = feature_df[feature].to_numpy()

    std = np.std(values, ddof=1)

    lower, upper = bootstrap_std_ci(
        values,
        n_bootstrap=10000,
        confidence=0.90,
        rng=rng
    )

    results.append({
        "Feature": feature,
        "Std": std,
        "90% CI Lower": lower,
        "90% CI Upper": upper
    })

std_ci_df = pd.DataFrame(results)

std_ci_df

,Feature,Std,90% CI Lower,90% CI Upper
0,min1,9.569975,8.248268,10.740930
1,max1,4.394362,3.312884,5.308434
2,mean1,5.335718,4.697892,5.884208
3,median1,5.440054,4.795507,6.000395
4,std1,1.772153,1.573009,1.948951
5,1st_quart1,6.153590,5.567201,6.655016
6,3rd_quart1,5.138925,4.337018,5.831949
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.628981,5.401845
9,mean2,1.574164,1.394395,1.704686


#### iv. Select Features

I would select **minimum, mean, and maximum** as the three most important time-domain features.

The **mean** represents the overall level of a time series, while the **minimum** and **maximum** capture its lower and upper extremes. Together, these features provide a simple summary of both the central tendency and range of values observed during an activity.

Although other features such as the median, quartiles, and standard deviation also contain useful information, minimum, mean, and maximum provide an interpretable representation of the overall level and extremes of each time series.

## 2. ISLR 3.7.4

### (a) Linear Train

**Cubic regression has lower (or at worst equal) training RSS**

The cubic regression will have a lower training RSS than the linear regression, or at worst the same training RSS. The cubic model is more flexible and contains the linear model as a special case when the coefficients of $X^2$ and  $X^3$ are zero. Therefore, it can fit the training data at least as well as the linear model.

### (b) Linear Test

**Linear regression is expected to have lower test RSS**

The linear regression is expected to have a lower test RSS. Since the true relationship is linear, the additional quadratic and cubic terms are unnecessary. The cubic model is more flexible and may fit random noise in the training data, resulting in higher variance and potentially worse generalization to test data.

### (c) Not Linear Train

**Cubic regression has lower (or equal) training RSS**

The cubic regression will have a lower training RSS than the linear regression, or at worst the same training RSS. Regardless of the true relationship, the cubic model is more flexible and contains the linear model as a special case, so adding the quadratic and cubic terms cannot increase the training RSS.

### (d) Not Linear Testing

**There is not enough information to determine which model has lower test RSS**

There is not enough information to determine which model will have the lower test RSS. If the true relationship is sufficiently nonlinear and is well approximated by a cubic function, the cubic regression may have lower test RSS because of its greater flexibility. However, if the nonlinearity is weak, the additional flexibility of the cubic model may increase variance and lead to a higher test RSS. Therefore, the result depends on the form of the true relationship.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 